# 04.2 Debugging Playbook / 常见报错与调试

这一节不追求“永远不报错”，而是训练你看到报错时知道从哪下手。  
The goal of this notebook is not "never make errors", but to train you to know where to start when an error appears.

重点概念 / Key concepts:

- 形状错误 / shape mismatch
- 数据类型错误 / dtype mismatch
- 设备错误 / device mismatch
- 数值不稳定 / numerical instability
- 调试 checklist / debugging checklist

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 识别几类最常见的训练报错 / Recognize several of the most common training errors.
2. 读懂 shape 和 dtype 相关报错 / Read shape- and dtype-related error messages.
3. 用最小打印信息快速定位问题 / Use minimal but useful prints to localize problems quickly.
4. 理解为什么数值不稳定会导致 `inf / nan` / Understand why numerical instability can produce `inf / nan`.
5. 积累一个可复用的调试流程 / Build a reusable debugging workflow.

In [ ]:
import traceback

import torch
import torch.nn as nn

torch.manual_seed(42)

## 1. 一个基本原则 / One Basic Principle

看到报错时，不要先大改代码。  
When you see an error, do not immediately rewrite large parts of the code.

先回答这几个问题 / First answer these questions:

- 输入 shape 是什么 / what is the input shape
- 输出 shape 是什么 / what is the output shape
- target 的 shape 和 dtype 是什么 / what are the target shape and dtype
- model 和 tensor 在同一个 device 上吗 / are the model and tensors on the same device
- loss 是有限值吗 / is the loss finite

In [ ]:
def print_tensor_info(name, tensor):
    print(
        f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, "
        f"device={tensor.device}, min={tensor.min().item():.4f}, max={tensor.max().item():.4f}"
    )


sample_x = torch.randn(4, 3)
sample_y = torch.tensor([0, 1, 0, 1], dtype=torch.long)
print_tensor_info("sample_x", sample_x)
print_tensor_info("sample_y", sample_y.float())

## 2. Shape Mismatch / 形状不匹配

这是最常见的一类错误。  
This is one of the most common error types.

典型原因 / Typical causes:

- `Linear` 的输入维度写错 / the input dimension of a `Linear` layer is wrong
- 忘了 flatten / forgot to flatten
- loss 的输入和 target 维度不匹配 / the loss input and target shapes do not match

In [ ]:
x = torch.randn(5, 3)
broken_linear = nn.Linear(4, 2)

try:
    broken_linear(x)
except Exception as e:
    print("Caught shape mismatch / 捕获到形状错误:")
    print(type(e).__name__)
    print(e)

In [ ]:
fixed_linear = nn.Linear(3, 2)
out = fixed_linear(x)
print_tensor_info("x", x)
print_tensor_info("out", out)
print("结论 / Conclusion: 最先检查最后一维是否和 in_features 对齐 / first check whether the last dimension matches in_features.")

## 3. Target Dtype 错误 / Target Dtype Mismatch

`CrossEntropyLoss` 的 target 通常要是 `long` 类型的类别索引。  
The target for `CrossEntropyLoss` is usually expected to be class indices of dtype `long`.

In [ ]:
logits = torch.randn(4, 3)
wrong_targets = torch.tensor([0.0, 1.0, 2.0, 1.0], dtype=torch.float32)
loss_fn = nn.CrossEntropyLoss()

try:
    loss_fn(logits, wrong_targets)
except Exception as e:
    print("Caught dtype mismatch / 捕获到 dtype 错误:")
    print(type(e).__name__)
    print(e)

In [ ]:
correct_targets = wrong_targets.long()
loss = loss_fn(logits, correct_targets)
print_tensor_info("logits", logits)
print("correct_targets dtype =", correct_targets.dtype)
print("loss =", float(loss))

## 4. BCEWithLogitsLoss 的 Shape 错误
## A Shape Error with BCEWithLogitsLoss

`BCEWithLogitsLoss` 要求 input 和 target 的 shape 一致。  
`BCEWithLogitsLoss` expects the input and target shapes to match.

In [ ]:
binary_logits = torch.randn(4, 1)
binary_targets_wrong = torch.tensor([1.0, 0.0, 1.0, 0.0])
bce_loss = nn.BCEWithLogitsLoss()

try:
    bce_loss(binary_logits, binary_targets_wrong)
except Exception as e:
    print("Caught BCE shape mismatch / 捕获到 BCE 形状错误:")
    print(type(e).__name__)
    print(e)

binary_targets_correct = binary_targets_wrong.unsqueeze(1)
fixed_loss = bce_loss(binary_logits, binary_targets_correct)
print("fixed_loss =", float(fixed_loss))
print("binary_logits.shape =", binary_logits.shape)
print("binary_targets_correct.shape =", binary_targets_correct.shape)

## 5. Device Mismatch / 设备不一致

最典型的情形是：  
The most typical case is:

- model 在 GPU / model is on GPU
- data 在 CPU / data is on CPU

如果当前机器没有可用 CUDA，这里就只演示检查方式。  
If CUDA is not available on the current machine, we only demonstrate the checking method.

In [ ]:
def assert_same_device(model, *tensors):
    model_device = next(model.parameters()).device
    for idx, tensor in enumerate(tensors):
        assert tensor.device == model_device, (
            f"tensor {idx} is on {tensor.device}, but model is on {model_device}"
        )


cpu_model = nn.Linear(3, 2)
cpu_x = torch.randn(2, 3)
assert_same_device(cpu_model, cpu_x)
print("CPU case passed / CPU 情况检查通过")

if torch.cuda.is_available():
    gpu_model = nn.Linear(3, 2).cuda()
    try:
        assert_same_device(gpu_model, cpu_x)
    except Exception as e:
        print("Simulated device mismatch / 模拟 device 错误:")
        print(type(e).__name__)
        print(e)
else:
    print("CUDA not available / 当前环境没有可用 CUDA，所以跳过真实 device mismatch 复现。")

## 6. Numerical Instability / 数值不稳定

`nan loss` 不一定来自模型结构本身，也可能来自不稳定的数值计算。  
`nan loss` does not always come from the model structure itself; it can also come from unstable numerical computation.

In [ ]:
large_scores = torch.tensor([1000.0, 1001.0])
naive_softmax = torch.exp(large_scores) / torch.exp(large_scores).sum()
stable_softmax = torch.softmax(large_scores, dim=0)

print("naive_softmax =", naive_softmax)
print("contains nan / 是否含 nan =", torch.isnan(naive_softmax).any().item())
print("stable_softmax =", stable_softmax)

In [ ]:
constant_vector = torch.tensor([3.0, 3.0, 3.0])
bad_normalized = (constant_vector - constant_vector.mean()) / constant_vector.std()
safe_normalized = (constant_vector - constant_vector.mean()) / (constant_vector.std() + 1e-6)

print("bad_normalized =", bad_normalized)
print("has nan / 是否有 nan =", torch.isnan(bad_normalized).any().item())
print("safe_normalized =", safe_normalized)

## 7. 一个实用的 Debug Checklist / A Practical Debug Checklist

每次训练前后，你都可以做这些小检查 / Before and during training, you can run these quick checks:

- 打印一个 batch 的 shape / print the shape of one batch
- 打印 target dtype / print the target dtype
- 检查 logits 和 target 是否对应 / check whether logits and targets match the expected loss function
- 检查 loss 是否是有限值 / check whether the loss is finite
- 检查梯度是不是 `None` 或全 0 / check whether gradients are `None` or all zeros

In [ ]:
debug_model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
debug_x = torch.randn(6, 3)
debug_y = torch.tensor([0, 1, 0, 1, 1, 0], dtype=torch.long)
debug_loss_fn = nn.CrossEntropyLoss()

logits = debug_model(debug_x)
loss = debug_loss_fn(logits, debug_y)
debug_model.zero_grad()
loss.backward()

print_tensor_info("debug_x", debug_x)
print_tensor_info("logits", logits)
print("debug_y dtype =", debug_y.dtype)
print("loss is finite / loss 是否有限 =", torch.isfinite(loss).item())

for name, param in debug_model.named_parameters():
    grad_norm = None if param.grad is None else float(param.grad.norm())
    print(f"{name}: grad_norm={grad_norm}")

In [ ]:
# 练习 1 / Exercise 1
# 如果 nn.Linear(16, 4) 收到输入 x.shape == (32, 8)，
# If nn.Linear(16, 4) receives x.shape == (32, 8),
# 你第一反应应该检查什么？
# what should be the first thing you check?

练习 1 参考答案 / Exercise 1 Reference Answer

先检查输入最后一维和 `in_features` 是否一致。  
First check whether the last input dimension matches `in_features`.

这里 `8 != 16`，所以线性层输入维度不匹配。  
Here `8 != 16`, so the linear layer input dimension is mismatched.

In [ ]:
# 练习 2 / Exercise 2
# 什么时候你应该怀疑是 dtype 错误，而不是 shape 错误？
# When should you suspect a dtype error instead of a shape error?

练习 2 参考答案 / Exercise 2 Reference Answer

当 shape 看起来是对的，但 loss 或算子明确要求某种类型时，就要先查 dtype。  
When the shapes look correct but the loss function or operator explicitly expects a specific type, check dtype first.

例如 `CrossEntropyLoss` 常见地要求 target 是 `long`。  
For example, `CrossEntropyLoss` commonly expects the target to be `long`.

## 8. 小结 / Summary

这一节最重要的不是记住所有报错文本，而是形成顺序化的排查习惯。  
The most important outcome of this notebook is not memorizing all error messages, but building an ordered debugging habit.

建议你先查这五件事 / I suggest checking these five things first:

1. shape
2. dtype
3. device
4. loss 是否有限 / whether the loss is finite
5. 梯度是否正常 / whether gradients look normal